# 🐍 Part 2: Database Enrichment & Data Wrangling via Python 
## Refactoring and Enriching the `Bookings` Table (`05_update_bookings.ipynb`)


## 🎯 Context & Objectives

The goal of this stage is to perform strategic data enrichment on the `bookings` table within the `insight_places` database using Python (`05_update_bookings.ipynb`). By expanding the volume of transactions, establishing an automated seasonal pricing model (`total_price`), enforcing temporal logic between booking creation (`created_at`) and stay dates, standardizing international transaction statuses (`booking_status`), and capturing financial channels (`payment_method`), I will transform this table into a high-density fact table for advanced analytical queries and BI dashboards. 

This notebook represents a core module of a sequential ETL execution, preserving strict relational integrity and cross-table business logic with the `listings` and `customers` tables.


## 🛠️ Environment Setup & Dependencies

To execute this ETL pipeline securely and efficiently, the following Python libraries are required: 
* **`sqlalchemy` & `pymysql`**: Database abstraction layer and MySQL driver for executing DDL/DML operations. 
* **`python-dotenv`**: Security protocol to parse database credentials from the existing `.env` environment file without exposing sensitive keys. 
* **`pandas`**: Data manipulation, aggregation, and structural validation. 
* **`random` & `datetime`**: Native modules used to generate realistic booking frequency, seasonal rate multipliers, and cross-table capacity validation. 


## 📌 Refining the `Bookings` Table 

This module focuses on upgrading booking transaction density, temporal rules, and financial attribute integrity: 
1. **Transaction Density Expansion:** Generating multiple realistic bookings per property to enable historical occupancy rate tracking and recurrence analysis. 
2. **Temporal Integrity Enforcer (`created_at`):** Establishing booking creation timestamps that strictly precede the check-in date (`start_date`), preserving realistic lead-time distributions. 
3. **Capacity-Constrained Guest Allocation (`guests_count`):** Enforcing cross-table business logic where the assigned number of guests strictly respects each property's maximum capacity (`max_guests` from `listings`). 
4. **Seasonal & Dynamic Revenue Calibration (`total_price`):** Calculating total booking revenue by applying seasonal multipliers (high, mid, and low season) and weekend surcharges directly onto each listing's base daily rate (`base_daily_rate`). 
5. **International Status Standardization (`booking_status`):** Standardizing operational booking states into English values (`Confirmed`, `Completed`, `Canceled`) to support conversion and churn metrics. 
6. **Financial Channel Mapping (`payment_method`):** Capturing payment modalities (`Credit Card`, `Debit Card`, `Cash`, `PayPal`, `Bank Transfer`) formatted for global platform contexts


### 1) Installing and importing the libraries
In this step, I will load the libraries that will be used throughout the project.


In [1]:
!pip install sqlalchemy
!pip install pymysql
import pandas as pd
import random
import datetime
from sqlalchemy import create_engine, text
import pymysql


#### 1.1) Installing the data protection library
I will install the python-dotenv library to ensure the security of the MySQL password.


In [2]:
!pip install python-dotenv

#### 1.2) Environment Setup and Security Validation

In [3]:
# Installing and Importing libraries
import os
from dotenv import load_dotenv

# Loading variables from the .env file
load_dotenv()

# Assigning credentials to variables using os.getenv
user = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
host = os.getenv('DB_HOST', 'localhost')
port = os.getenv('DB_PORT', '3306')
database = os.getenv('DB_NAME', 'insight_places')

# Security Check: Validating if environment variables are present
if all([user, password, host, database]):
    print("🔒 Environment variables loaded successfully.")
    print(" System ready for database connection.") 
else: print("❌ Error: One or more credentials are missing from the .env file.")


🔒 Environment variables loaded successfully.
 System ready for database connection.


#### 1.3) Database Engine Initialization
Using the validated credentials, I will use SQLAlchemy to create the engine. This object will be the communication bridge between Python and MySQL, allowing for efficient data transfer.


In [4]:
# Creating the database connection engine using validated credentials

from urllib.parse import quote_plus

# Encodes the password to prevent URL syntax errors with the '@' character
safe_password = quote_plus(password)

engine = create_engine(f"mysql+pymysql://{user}:{safe_password}@{host}:{port}/{database}")
print("🚀 Engine connection is ready!")


🚀 Engine connection is ready!


## 2) Execution: Refining and Enriching the `Bookings` Table

This section executes the step-by-step transformation pipeline to expand booking volume, enforce cross-table guest limits, apply dynamic seasonal revenue calculations, and standardize financial and operational attributes.


### 2.1 Schema Modernization (DDL Adjustments)

In this step, I modify the `bookings` table structure by verifying and adding missing attributes (`created_at`, `guests_count`, `booking_status`, `payment_method`) to ensure the table supports enriched analytical metrics, operational statuses, and seasonal pricing logic.


In [5]:
def update_bookings_schema():
    with engine.begin() as conn:
        print("Checking existing columns in 'bookings' table...")
        
        # Inspection of current columns
        columns_query = text("""
            SELECT COLUMN_NAME 
            FROM INFORMATION_SCHEMA.COLUMNS 
            WHERE TABLE_SCHEMA = DATABASE() AND TABLE_NAME = 'bookings'
        """)
        existing_columns = [row[0] for row in conn.execute(columns_query).fetchall()]
        
        # Dictionary of required new columns and their SQL definitions
        columns_to_add = {
            'created_at': 'DATE',
            'guests_count': 'INT',
            'booking_status': 'VARCHAR(20)',
            'payment_method': 'VARCHAR(20)'
        }
        
        # Add missing columns
        for col_name, col_type in columns_to_add.items():
            if col_name not in existing_columns:
                conn.execute(text(f"ALTER TABLE bookings ADD COLUMN {col_name} {col_type};"))
                print(f"  `{col_name}` column successfully added.")
            else:
                print(f"  `{col_name}` column already exists.")

        print("✅ Bookings schema modernization completed successfully!")

update_bookings_schema()


Checking existing columns in 'bookings' table...
  `created_at` column successfully added.
  `guests_count` column successfully added.
  `booking_status` column successfully added.
  `payment_method` column successfully added.
✅ Bookings schema modernization completed successfully!


### 2.2 Synthetic Data Generation & Business Logic Enrichment

In this step, I generate an enriched dataset of bookings for all properties in the `listings` table. The script enforces business rules including capacity boundaries (`guests_count` $\le$ `max_guests`), seasonal daily rates, weekend multipliers, dynamic booking statuses (`Confirmed`, `Completed`, `Canceled`), lead-time constraints for `created_at`, and global payment methods.


In [6]:
import numpy as np
from datetime import datetime, timedelta

def populate_enriched_bookings():
    with engine.begin() as conn:
        print("📥 Fetching listings and customers data...")
        
        # 1. Fetch available listings with their capacities and daily rates
        listings_df = pd.read_sql("""
            SELECT listing_id, base_daily_rate, max_guests 
            FROM listings 
            WHERE is_active = 1 OR is_active IS NULL
        """, conn)
        
        # 2. Fetch available customers
        customers_df = pd.read_sql("SELECT customer_id FROM customers", conn)
        
        if listings_df.empty or customers_df.empty:
            print("❌ Error: 'listings' or 'customers' table is empty. Please check database state.")
            return

        customer_ids = customers_df['customer_id'].tolist()
        payment_methods = ['Credit Card', 'Debit Card', 'Cash', 'PayPal', 'Bank Transfer']
        status_options = ['Completed', 'Completed', 'Completed', 'Confirmed', 'Canceled'] # Weighted distribution
        
        new_bookings = []
        booking_counter = 1001

        # Date range for booking generation (covering historical and upcoming periods)
        start_range = datetime(2023, 1, 1)
        end_range = datetime(2025, 12, 31)
        total_days = (end_range - start_range).days

        print("⚡ Generating synthetic bookings with business logic...")

        for _, listing in listings_df.iterrows():
            l_id = listing['listing_id']
            base_rate = float(listing['base_daily_rate']) if listing['base_daily_rate'] else 150.0
            max_g = int(listing['max_guests']) if listing['max_guests'] else 2

            # Generate between 5 to 12 bookings per listing to expand density
            num_bookings = random.randint(5, 12)

            for _ in range(num_bookings):
                # Random start date
                random_days = random.randint(0, total_days)
                b_start = start_range + timedelta(days=random_days)
                
                # Stay duration between 1 and 14 days
                stay_duration = random.randint(1, 14)
                b_end = b_start + timedelta(days=stay_duration)

                # Lead time: created_at between 1 and 60 days BEFORE start_date
                lead_time = random.randint(1, 60)
                created_at = b_start - timedelta(days=lead_time)

                # 1. Guest count respects max_guests
                guests_count = random.randint(1, max_g)

                # 2. Seasonal pricing logic
                month = b_start.month
                if month in [12, 1, 2, 7]:  # High season (Summer / July holidays)
                    season_factor = random.uniform(1.30, 1.55)
                elif month in [3, 4, 5, 9, 10, 11]:  # Mid season
                    season_factor = random.uniform(1.00, 1.15)
                else:  # Low season
                    season_factor = random.uniform(0.85, 0.95)

                # Weekend surcharge check (Friday, Saturday, Sunday)
                has_weekend = any((b_start + timedelta(days=i)).weekday() in [4, 5, 6] for i in range(stay_duration))
                weekend_factor = 1.10 if has_weekend else 1.00

                # Final daily price calculation
                effective_daily_rate = base_rate * season_factor * weekend_factor
                total_price = round(effective_daily_rate * stay_duration, 2)

                # Attributes mapping
                b_status = random.choice(status_options)
                p_method = random.choice(payment_methods)
                cust_id = random.choice(customer_ids)

                new_bookings.append({
                    'booking_id': f"BK_{booking_counter}",
                    'customer_id': cust_id,
                    'listing_id': l_id,
                    'start_date': b_start.strftime('%Y-%m-%d'),
                    'end_date': b_end.strftime('%Y-%m-%d'),
                    'created_at': created_at.strftime('%Y-%m-%d'),
                    'guests_count': guests_count,
                    'total_price': total_price,
                    'booking_status': b_status,
                    'payment_method': p_method
                })
                
                booking_counter += 1

        # Convert to DataFrame
        enriched_bookings_df = pd.DataFrame(new_bookings)

        # Clear legacy single-row bookings and insert enriched dataset
        print("🧹 Truncating legacy 'bookings' table...")
        conn.execute(text("DELETE FROM bookings;"))

        print(f"🚀 Inserting {len(enriched_bookings_df)} enriched booking records into MySQL...")
        enriched_bookings_df.to_sql('bookings', con=conn, if_exists='append', index=False)

        print("✅ Data enrichment and populate pipeline executed successfully!")

populate_enriched_bookings()


📥 Fetching listings and customers data...
⚡ Generating synthetic bookings with business logic...
🧹 Truncating legacy 'bookings' table...
🚀 Inserting 72095 enriched booking records into MySQL...
✅ Data enrichment and populate pipeline executed successfully!


### 2.3 Data Quality Check & Structural Inspection

In this final step, I execute verification queries to confirm row count, inspect sample records, validate table relationships, and verify that all business constraints (such as guest limits and temporal sequences) were correctly applied.


In [7]:
def validate_bookings_data():
    with engine.begin() as conn:
        print("🔍 Executing Data Quality Inspection on 'bookings' table...\n")
        
        # 1. Total row count check
        count_result = conn.execute(text("SELECT COUNT(*) FROM bookings;")).scalar()
        print(f"📊 Total Records in 'bookings': {count_result} rows.")
        
        # 2. Status distribution breakdown
        status_df = pd.read_sql("""
            SELECT booking_status, COUNT(*) as total_count, ROUND(AVG(total_price), 2) as avg_price 
            FROM bookings 
            GROUP BY booking_status;
        """, conn)
        print("\n📈 Booking Status Breakdown & Average Revenue:")
        print(status_df.to_string(index=False))
        
        # 3. Capacity integrity check (Verifying if guests_count exceeds max_guests)
        overflow_check = conn.execute(text("""
            SELECT COUNT(*) 
            FROM bookings b 
            JOIN listings l ON b.listing_id = l.listing_id 
            WHERE b.guests_count > l.max_guests;
        """)).scalar()
        
        if overflow_check == 0:
            print("\n✅ Capacity Rule Passed: 0 bookings exceed maximum property capacity.")
        else:
            print(f"\n⚠️ Capacity Rule Failed: {overflow_check} records violate capacity limits.")

        # 4. Sample record preview
        print("\n👀 Preview of 5 Enriched Booking Records:")
        sample_df = pd.read_sql("SELECT * FROM bookings LIMIT 5;", conn)
        print(sample_df.to_string(index=False))

validate_bookings_data()


🔍 Executing Data Quality Inspection on 'bookings' table...

📊 Total Records in 'bookings': 72095 rows.

📈 Booking Status Breakdown & Average Revenue:
booking_status  total_count  avg_price
     Confirmed        14491    4210.62
     Completed        43216    4239.80
      Canceled        14388    4287.16

✅ Capacity Rule Passed: 0 bookings exceed maximum property capacity.

👀 Preview of 5 Enriched Booking Records:
booking_id customer_id listing_id start_date   end_date  total_price created_at  guests_count booking_status payment_method
  BK_10000        1486       2103 2024-06-15 2024-06-22      2144.88 2024-05-14             4      Confirmed     Debit Card
  BK_10001        4900       2103 2024-11-01 2024-11-08      2670.50 2024-09-07             4      Completed           Cash
  BK_10002        7805       2103 2025-02-13 2025-02-26      6558.65 2025-02-06             3      Completed         PayPal
  BK_10003        1377       2103 2023-11-26 2023-11-29      1113.93 2023-09-27       